# EuroSAT: DINOv2 Partial Fine-Tuning

Optional supervised adaptation of **DINOv2 ViT-S/14**, separate from `EuroSAT_Representation_Benchmark.ipynb`. Update the **last Transformer block, final LayerNorm and ten-class head**; freeze the patch embedding, positional/class tokens and earlier blocks.

We compare the same pretrained model and neural classification head under **frozen-backbone** and **last-block fine-tuning** conditions. Both receive identical original supports, fixed augmentation views, head initialization and batch order. This control isolates the adaptation effect more directly than comparing a neural head against the main notebook's logistic regression.

The proposal treats fine-tuning as an optional separate experiment. All ten classes, original 60/20/20 splits and nested 1/5/10-shot supports are reused. Development labels choose the epoch; held-out labels are used only after training. Saved smoke outputs validate execution and are not final findings.


## Research direction: how much should we adapt?

**Research question:** can adapting DINOv2's pretrained representation improve few-shot satellite-image classification more than augmenting support images alone? The reported small augmentation gains and larger gains from additional originals motivate this question; they do not establish that fine-tuning will help.

| Direction | What changes | Decision |
|---|---|---|
| Partial fine-tuning | Last 1 block + final LayerNorm + head | Priority 1; implemented and selected |
| Two-block ablation | Last 2 blocks + final LayerNorm + head | Implemented; optional sensitivity study |
| LoRA / adapters | Small additional trainable modules | Future comparison; not implemented |
| Normalization-only adaptation | Final LayerNorm + head | Future smaller-scope control; not implemented |

**Why one block first?** The hypothesis is that late representations can adapt to satellite imagery while limiting updated weights. This is a conservative scope relative to two blocks, not a proven optimum. The head-only baseline uses the same initialization and optimizer. Full-backbone training is disabled.

**Decision rule:** fix the one-block protocol before the final study. Set `LAST_BLOCKS = 2` only for a separately reported ablation; do not choose the better scope from query scores. Learning rates are starting settings. Report negative or inconsistent gains, especially at 1 shot.

Background: [DINOv2 paper](https://arxiv.org/abs/2304.07193) and [official implementation](https://github.com/facebookresearch/dinov2). Neither establishes that this EuroSAT adaptation will outperform frozen features.

### Alignment with the project proposal

The [proposal](docs/project-proposal.pdf), **page 4**, explicitly permits fine-tuning as an optional extension reported separately from frozen experiments. The primary benchmark remains the four frozen representations with common logistic regression.

| Proposal commitment | This extension |
|---|---|
| DINOv2 ViT-S/14; exact checkpoint (p. 1) | Same official pinned pretrained backbone |
| Ten classes, deduplication, 60/20/20 split, nested 1/5/10 shots (p. 3–4) | Same manifests and original support samples |
| Support-only augmentation and repeated-original control (p. 4–5) | None / repeat / flip crossed with frozen / adapted backbone |
| Original + four views versus five originals (p. 5) | Paired 1+4 versus 5-original contrast in full mode |
| Accuracy, Macro-F1, class recall, confusion matrices and cost (p. 5) | Classification metrics, parameter counts and training time |

This **2 × 3 extension** runs two adaptation conditions × three support treatments × three shot budgets × 20 full trials = **360 fits**. Flip is a prespecified proposal-listed policy, not a query-selected winner. Repeat and flip both add four rows per original in full mode; only flip changes pixels. The primary notebook retains the broader augmentation and nine-view experiments.

The neural-head baseline differs from the primary logistic-regression procedure. Within this notebook, the head, optimizer and original supports are matched to isolate adaptation. Cross-notebook differences cannot be attributed solely to fine-tuning. This follow-up is exploratory because existing query results motivated it; it is not a new independent confirmation set.


## 1. Runtime and model

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

TARGET = os.environ.get('EUROSAT_TARGET', 'colab_gpu')  # Set local for a local VS Code kernel.
MODE = os.environ.get('EUROSAT_FINETUNE_MODE', 'smoke')
MODEL = 'dinov2_vits14'  # This extension focuses on DINOv2.
LAST_BLOCKS = int(os.environ.get('EUROSAT_FINETUNE_BLOCKS', '1'))  # Optional ablation: 2.
if TARGET not in {'colab_gpu', 'local'} or MODE not in {'smoke', 'full'}:
    raise ValueError('Choose TARGET colab_gpu/local and MODE smoke/full')
print('Selected model:', MODEL, '| mode:', MODE)


## 2. Dependencies and shared storage

In [ ]:
import importlib.util

requirements = {'torch': 'torch>=2.6,<3', 'torchvision': 'torchvision>=0.21,<1',
                'numpy': 'numpy>=1.26,<3', 'sklearn': 'scikit-learn>=1.4,<1.8',
                'PIL': 'Pillow>=10,<13', 'pandas': 'pandas>=2.2,<3',
                'matplotlib': 'matplotlib>=3.8,<4', 'seaborn': 'seaborn>=0.13,<0.14'}
missing = [package for module, package in requirements.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', *missing])
if TARGET == 'colab_gpu':
    from google.colab import drive
    drive.mount('/content/drive')
    parent = Path('/content/drive/MyDrive/ColabNotebooks/cs7643')
    previous = parent / 'EuroSATLowData'
    default = previous if previous.exists() else parent / 'EuroSATRepresentationBenchmark'
    STORAGE = Path(os.environ.get('EUROSAT_STORAGE', str(default)))
    SCRATCH = Path('/content/eurosat-representation-benchmark')
else:
    current = Path.cwd()
    project = next((p for p in [current, *current.parents] if (p / 'pyproject.toml').exists()), current)
    STORAGE = Path(os.environ.get('EUROSAT_STORAGE', str(project)))
    SCRATCH = STORAGE / 'data/extracted'
STORAGE.mkdir(parents=True, exist_ok=True)
print('Shared dataset/weights:', STORAGE)
print('Separate adaptation checkpoints: cache/finetuning; results: outputs/finetuning')


## 3. Actual implementation

Run the helper cells in order. Imports appear once; only the required shared helpers are included. All code is visible and generated from `src/eurosat/`, so fixes have one source of truth. Training lives in `finetuning.py`; tables and plots live in `finetuning_reporting.py`. Rebuild with `python scripts/build_finetuning_notebook.py` after edits.


### Imports

In [ ]:
from dataclasses import dataclass, field, replace, asdict
import math
import hashlib
import json
import tempfile
import numpy as np
import sklearn
import torch
import torchvision
import random
import shutil
import urllib.request
import zipfile
import time
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from torchvision.transforms import functional as TF, InterpolationMode
from torch import nn
from torchvision import models
from torch.utils.data import DataLoader, Dataset, ConcatDataset
from sklearn.metrics import confusion_matrix

### Config

In [ ]:
PIPELINE_VERSION = '1'
DATA_URL = ('https://huggingface.co/datasets/torchgeo/eurosat/resolve/'
            'c877bcd43f099cd0196738f714544e355477f3fd/EuroSAT.zip')
DATA_MD5 = 'c8fa014336c82ac7804f0398fcb19387'
DINO_REVISION = '7764ea0f912e53c92e82eb78a2a1631e92725fc8'
CLASSES = ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
           'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
MODEL_IDS = {
    'resnet18': 'torchvision.ResNet18_Weights.IMAGENET1K_V1',
    'convnext_tiny': 'torchvision.ConvNeXt_Tiny_Weights.IMAGENET1K_V1',
    'vit_b_16': 'torchvision.ViT_B_16_Weights.IMAGENET1K_V1',
    'dinov2_vits14': 'facebookresearch/dinov2:' + DINO_REVISION + '/dinov2_vits14',
}
POLICIES = ['none', 'repeat', 'flip', 'rotation', 'translation', 'scale',
            'brightness_contrast', 'noise', 'blur', 'erasing']
OPTIONAL_POLICIES = ['cutout']


@dataclass
class Config:
    mode: str = 'smoke'
    seed: int = 42
    models: list = field(default_factory=lambda: ['resnet18'])
    policies: list = field(default_factory=lambda: ['none', 'repeat', 'flip'])
    shots: list = field(default_factory=lambda: [1, 5, 10])
    trials: int = 1
    views: int = 1
    per_class_limit: int | None = 40
    batch_size: int = 32
    chunk_size: int = 256
    policy_strengths: dict = field(default_factory=dict)
    classifier_cs: list = field(default_factory=lambda: [0.1, 1.0, 10.0])

    @classmethod
    def full(cls):
        return cls(mode='full', models=list(MODEL_IDS), policies=POLICIES.copy(),
                   trials=20, views=4, per_class_limit=None)

    def validate(self):
        if self.mode not in {'smoke', 'full'}:
            raise ValueError('mode must be smoke or full')
        if not self.models or set(self.models) - MODEL_IDS.keys():
            raise ValueError('Choose supported models: ' + ', '.join(MODEL_IDS))
        if 'none' not in self.policies or set(self.policies) - set(POLICIES + OPTIONAL_POLICIES):
            raise ValueError('Policies must include none and contain supported names')
        if not self.shots or min(self.shots) < 1 or min(self.trials, self.views, self.batch_size, self.chunk_size) < 1:
            raise ValueError('Shot budgets, trials, views and batch/chunk sizes must be positive')
        if any(len(values) != len(set(values)) for values in [self.models, self.policies, self.shots]):
            raise ValueError('Models, policies and shot budgets must be unique')
        if set(self.policy_strengths) - set(POLICIES[2:] + OPTIONAL_POLICIES):
            raise ValueError('Strengths must name supported augmentation policies')
        if any(not math.isfinite(s) or not 0 < s <= 2 for s in self.policy_strengths.values()):
            raise ValueError('Augmentation strengths must be finite and in (0, 2]')
        if not self.classifier_cs or min(self.classifier_cs) <= 0:
            raise ValueError('Classifier C values must be positive')

### Cache

In [ ]:
def key(value):
    text = json.dumps({'pipeline': PIPELINE_VERSION, 'value': value},
                      sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(text.encode()).hexdigest()


def digest_file(path, algorithm='sha256'):
    result = hashlib.new(algorithm)
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()


def atomic(path, write):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix='.' + path.name, dir=path.parent)
    os.close(fd)
    try:
        write(Path(temporary))
        os.replace(temporary, path)
    finally:
        Path(temporary).unlink(missing_ok=True)


def write_json(path, value):
    atomic(path, lambda p: p.write_text(json.dumps(value, indent=2, sort_keys=True,
                                                   allow_nan=False) + '\n'))




def prepare_storage(storage):
    import torch
    storage = Path(storage)
    for name in ['data', 'cache/weights', 'cache/features', 'cache/experiments', 'outputs']:
        (storage / name).mkdir(parents=True, exist_ok=True)
    torch.hub.set_dir(str(storage / 'cache/weights/torch'))
    return storage

### Runtime

In [ ]:
def environment():
    return {'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
            'numpy': np.__version__, 'sklearn': sklearn.__version__}


def check_device(target='colab_gpu'):
    if target not in {'colab_gpu', 'local'}:
        raise ValueError('target must be colab_gpu or local')
    if target == 'colab_gpu':
        if not Path('/content').is_dir() or not torch.cuda.is_available():
            raise RuntimeError('Select a Colab GPU server (T4 or L4), not a local/CPU/TPU '
                               'kernel. In VS Code: Select Another Kernel > Colab > '
                               'New Colab Server > GPU > Python 3 (ipykernel).')
        device = torch.device('cuda')
        label = torch.cuda.get_device_name(0)
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else
                              'mps' if torch.backends.mps.is_available() else 'cpu')
        label = str(device)
    if device.type == 'cpu':
        torch.set_num_threads(min(4, os.cpu_count() or 1))
    if (torch.ones(1, device=device) + 1).cpu().item() != 2:
        raise RuntimeError('Device operation check failed')
    print(f'Compute: {label} [{device}] â€” device check passed', flush=True)
    return device

### Augment

In [ ]:
def image_tensor(path, policy, seed, strength=1.):
    rng = random.Random(seed)
    with Image.open(path) as image:
        image = TF.resize(image.convert('RGB'), [224, 224], InterpolationMode.BILINEAR)
    if policy == 'flip':
        image = TF.hflip(image) if rng.random() < .5 else TF.vflip(image)
    elif policy == 'rotation':
        image = TF.rotate(image, rng.uniform(-30 * strength, 30 * strength), InterpolationMode.BILINEAR)
    elif policy == 'translation':
        image = TF.affine(image, 0, [rng.randint(-round(22 * strength), round(22 * strength)), rng.randint(-round(22 * strength), round(22 * strength))], 1., [0., 0.],
                          InterpolationMode.BILINEAR)
    elif policy == 'scale':
        image = TF.affine(image, 0, [0, 0], rng.uniform(1 - .2 * strength, 1 + .2 * strength), [0., 0.], InterpolationMode.BILINEAR)
    elif policy == 'brightness_contrast':
        image = TF.adjust_contrast(TF.adjust_brightness(image, rng.uniform(1 - .3 * strength, 1 + .3 * strength)), rng.uniform(1 - .3 * strength, 1 + .3 * strength))
    elif policy == 'blur':
        image = TF.gaussian_blur(image, [5, 5], [rng.uniform(.1, 1.5 * strength)] * 2)
    elif policy not in {'none', 'noise', 'erasing', 'cutout'}:
        raise ValueError('Unsupported image augmentation: ' + policy)
    tensor = TF.to_tensor(image)
    if policy == 'noise':
        generator = torch.Generator().manual_seed(seed)
        tensor = (tensor + torch.randn(tensor.shape, generator=generator) * (.05 * strength)).clamp(0, 1)
    elif policy in {'erasing', 'cutout'}:
        size = rng.randint(round(22 * strength ** .5), round(67 * strength ** .5))
        top, left = rng.randrange(225 - size), rng.randrange(225 - size)
        tensor[:, top:top + size, left:left + size] = .5 if policy == 'erasing' else 0.
    return TF.normalize(tensor, [0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

### Data

In [ ]:
def prepare_data(storage, scratch, config):
    """Persist one archive in Drive; read extracted images from fast runtime-local disk."""
    archive = storage / 'data/EuroSAT.zip'
    if not archive.exists():
        def download(temporary):
            print('Downloading EuroSAT RGB (~90 MB), once per storage folder...', flush=True)
            request = urllib.request.Request(DATA_URL, headers={'User-Agent': 'eurosat-representation-benchmark/0.1'})
            with urllib.request.urlopen(request, timeout=120) as response, temporary.open('wb') as stream:
                shutil.copyfileobj(response, stream)
            if digest_file(temporary, 'md5') != DATA_MD5:
                raise ValueError('Downloaded EuroSAT checksum does not match; rerun to retry')
        atomic(archive, download)
    if digest_file(archive, 'md5') != DATA_MD5:
        raise ValueError(f'EuroSAT archive is corrupt: {archive}')
    archive_id = digest_file(archive)
    raw = Path(scratch) / archive_id[:16]
    marker = raw / '.complete'
    if not marker.exists():
        raw.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as bundle:
            for name in bundle.namelist():
                if not (raw / name).resolve().is_relative_to(raw.resolve()):
                    raise ValueError('Archive contains an invalid path')
            bundle.extractall(raw)
        marker.write_text(archive_id)
    candidates = [raw] + sorted(p for p in raw.rglob('*') if p.is_dir())
    image_root = next((p for p in candidates if all((p / name).is_dir() for name in CLASSES)), None)
    if image_root is None:
        raise ValueError('EuroSAT class folders not found')
    manifest_path = storage / 'cache' / ('dataset-' + archive_id + '.json')
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text())
    else:
        records, duplicates, seen = [], [], {}
        for label, name in enumerate(CLASSES):
            for path in sorted((image_root / name).glob('*.jpg')):
                with Image.open(path) as image:
                    rgb = image.convert('RGB')
                    fingerprint = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                relative = path.relative_to(image_root).as_posix()
                if fingerprint in seen:
                    previous = seen[fingerprint]
                    if previous['label'] != label:
                        raise ValueError('Identical pixels have conflicting class labels')
                    duplicates.append({'kept': previous['path'], 'removed': relative})
                    continue
                record = {'path': relative, 'label': label, 'pixels_sha256': fingerprint}
                seen[fingerprint] = record
                records.append(record)
        manifest = {'archive_sha256': archive_id, 'records': records, 'duplicates': duplicates}
        write_json(manifest_path, manifest)
    records = manifest['records']
    if config.per_class_limit is not None:
        rng = random.Random(config.seed)
        records = [r for label in range(len(CLASSES)) for r in rng.sample(
            [r for r in records if r['label'] == label],
            min(config.per_class_limit, sum(r['label'] == label for r in records)))]
        records.sort(key=lambda r: r['path'])
    split = stratified_split(records, config.seed)
    dataset_id = key({'records': records, 'split': split})
    write_json(storage / 'cache' / ('split-' + dataset_id + '.json'),
               {'classes': CLASSES, 'records': records, 'split': split,
                'duplicate_images_removed': len(manifest['duplicates'])})
    counts = {name: len(indices) for name, indices in split.items()}
    print(f'Data: {len(records)} unique images, splits {counts}; '
          f'{len(manifest["duplicates"])} exact-pixel duplicates removed', flush=True)
    return image_root, records, split, dataset_id


def stratified_split(records, seed):
    split = {'support': [], 'dev': [], 'test': []}
    rng = random.Random(seed)
    for label in range(len(CLASSES)):
        indices = [i for i, row in enumerate(records) if row['label'] == label]
        rng.shuffle(indices)
        train, dev = int(len(indices) * .6), int(len(indices) * .2)
        if min(train, dev, len(indices) - train - dev) < 1:
            raise ValueError('Each class needs enough distinct images for three partitions')
        split['support'].extend(indices[:train])
        split['dev'].extend(indices[train:train + dev])
        split['test'].extend(indices[train + dev:])
    return split


def make_trials(records, split, config):
    trials = []
    maximum = max(config.shots)
    for trial in range(config.trials):
        rng = random.Random(config.seed + trial)
        selected = {}
        for label in range(len(CLASSES)):
            pool = [i for i in split['support'] if records[i]['label'] == label]
            if len(pool) < maximum:
                raise ValueError(f'Class {CLASSES[label]} has fewer than {maximum} support-pool images')
            # Shuffle the whole pool so changing the requested shot/view grid cannot
            # change the original support images in a corresponding trial.
            rng.shuffle(pool)
            selected[label] = pool[:maximum]
        trials.append({str(k): [i for label in range(len(CLASSES)) for i in selected[label][:k]]
                       for k in sorted(config.shots)})
    return trials

### Models

In [ ]:
def load_backbone(name, device):
    if name == 'resnet18':
        model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        model.fc = nn.Identity()
    elif name == 'convnext_tiny':
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        model.classifier[-1] = nn.Identity()
    elif name == 'vit_b_16':
        model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        model.heads = nn.Identity()
    elif name == 'dinov2_vits14':
        # Official implementation + weights pinned to a specific source revision.
        os.environ['XFORMERS_DISABLED'] = '1'
        model = torch.hub.load('facebookresearch/dinov2:' + DINO_REVISION,
                               'dinov2_vits14', pretrained=True,
                               trust_repo=True, skip_validation=True)
    else:
        raise ValueError('Unknown backbone: ' + name)
    model.requires_grad_(False)
    return model.to(device).eval()

### Features

In [ ]:
class Images(Dataset):
    def __init__(self, root, records, items, policy, seed, strength=1.):
        self.root, self.records, self.items = root, records, items
        self.policy, self.seed, self.strength = policy, seed, strength

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        image_id, view = self.items[index]
        record = self.records[image_id]
        # Image/view seeds are shared across backbones and independent of batch/chunk order.
        seed = int(key([self.seed, record['pixels_sha256'], self.policy, view])[:15], 16)
        return image_tensor(self.root / record['path'], self.policy, seed, self.strength), record['label']

### Evaluation

In [ ]:
def metrics_from_confusion(matrix):
    """Compute class metrics directly from counts, including old cached results."""
    cm = np.asarray(matrix, dtype=float)
    if cm.shape != (len(CLASSES), len(CLASSES)) or not np.isfinite(cm).all() or np.any(cm < 0) or np.any(cm != np.floor(cm)):
        raise ValueError('Expected a finite nonnegative ten-class count matrix')
    total = cm.sum()
    if total == 0:
        raise ValueError('Cannot score an empty query set')
    support, predicted, correct = cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    divide = lambda a, b: np.divide(a, b, out=np.zeros_like(a, dtype=float), where=b != 0)
    precision, recall = divide(correct, predicted), divide(correct, support)
    f1 = divide(2 * precision * recall, precision + recall)
    accuracy = correct.sum() / total
    expected = np.dot(support, predicted) / total ** 2
    denominator = np.sqrt((total ** 2 - np.dot(support, support)) *
                          (total ** 2 - np.dot(predicted, predicted)))
    return {'accuracy': float(accuracy), 'macro_f1': float(f1.mean()),
            'macro_precision': float(precision.mean()), 'macro_recall': float(recall.mean()),
            'weighted_f1': float(np.dot(f1, support) / total),
            'balanced_accuracy': float(recall[support > 0].mean()),
            'mcc': float((correct.sum() * total - np.dot(support, predicted)) / denominator) if denominator else 0.,
            'cohen_kappa': float((accuracy - expected) / (1 - expected)) if expected < 1 else None,
            'class_precision': precision.tolist(), 'class_recall': recall.tolist(),
            'class_f1': f1.tolist(), 'class_support': support.astype(int).tolist(),
            'confusion_matrix': cm.astype(int).tolist()}


def score(labels, predictions):
    matrix = confusion_matrix(labels, predictions, labels=list(range(len(CLASSES))))
    return metrics_from_confusion(matrix)

### Exploration

In [ ]:
STYLE = {'font.family': 'DejaVu Sans', 'font.size': 9, 'axes.titlesize': 9,
         'axes.labelsize': 9, 'axes.spines.top': False, 'axes.spines.right': False,
         'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.facecolor': 'white'}






def save_figure(figure, output, name):
    for extension in ['png', 'pdf']:
        figure.savefig(output / f'{name}.{extension}', dpi=300, bbox_inches='tight', pad_inches=.04)
    plt.close(figure)

### Fine tuning settings

In [ ]:
FINETUNE_VERSION = '1'
FEATURE_DIMENSIONS = {'resnet18': 512, 'convnext_tiny': 768, 'vit_b_16': 768, 'dinov2_vits14': 384}


@dataclass
class FineTuneConfig:
    model: str = 'dinov2_vits14'
    mode: str = 'smoke'
    seed: int = 42
    shots: list = field(default_factory=lambda: [1, 5, 10])
    trials: int = 1
    epochs: int = 2
    batch_size: int = 16
    patience: int = 3
    head_lr: float = .001
    backbone_lr: float = .00001
    weight_decay: float = .0001
    views: int = 1
    policy: str = 'flip'
    strategy: str = 'last_block'
    include_frozen_control: bool = True
    last_blocks: int = 1

    @classmethod
    def full(cls, model='dinov2_vits14'):
        return cls(model=model, mode='full', trials=20, epochs=10, views=4)

    def validate(self):
        if self.model not in MODEL_IDS or self.mode not in {'smoke', 'full'}:
            raise ValueError('Choose a supported model and smoke/full mode')
        if self.strategy != 'last_block':
            raise ValueError('Only partial last-block fine-tuning is enabled')
        if self.last_blocks not in {1, 2} or (self.model != 'dinov2_vits14' and self.last_blocks != 1):
            raise ValueError('Choose one final stage, or one/two blocks for DINOv2')
        if not self.shots or min(self.shots) < 1 or len(set(self.shots)) != len(self.shots):
            raise ValueError('Choose unique positive original-shot budgets')
        if min(self.trials, self.epochs, self.batch_size, self.patience) < 1 or self.views < 0:
            raise ValueError('Trials, epochs, batch size and patience must be positive; views nonnegative')
        if not all(np.isfinite(v) for v in [self.head_lr, self.backbone_lr, self.weight_decay]):
            raise ValueError('Optimizer settings must be finite')
        if min(self.head_lr, self.backbone_lr) <= 0 or self.weight_decay < 0:
            raise ValueError('Learning rates must be positive and weight decay nonnegative')
        if self.policy not in {'none', 'repeat', 'flip', 'rotation', 'translation', 'scale',
                               'brightness_contrast', 'noise', 'blur', 'erasing', 'cutout'}:
            raise ValueError('Unsupported training augmentation')

    def data_config(self):
        return Config(mode=self.mode, seed=self.seed, models=[self.model], shots=self.shots,
                      trials=self.trials, per_class_limit=40 if self.mode == 'smoke' else None)

### Partial model

In [ ]:
class AdaptedClassifier(nn.Module):
    def __init__(self, backbone, dimension):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(dimension, len(CLASSES))

    def forward(self, images):
        # Same representation normalization in the matched control and fine-tuned model.
        return self.head(nn.functional.normalize(self.backbone(images).flatten(1), dim=1))


def build_adapted_model(config, strategy, device, seed):
    """Use identical pretrained weights and head initialization across conditions."""
    config.validate()
    if strategy not in {'last_block', 'frozen_head'}:
        raise ValueError('Choose partial last_block adaptation or the frozen_head control')
    torch.manual_seed(seed)
    backbone = load_backbone(config.model, device)
    torch.manual_seed(seed)
    model = AdaptedClassifier(backbone, FEATURE_DIMENSIONS[config.model]).to(device)
    if strategy == 'last_block':
        if config.model == 'resnet18':
            blocks = [backbone.layer4]
        elif config.model == 'convnext_tiny':
            blocks = [backbone.features[-1], backbone.classifier[0]]
        elif config.model == 'vit_b_16':
            blocks = [backbone.encoder.layers[-1], backbone.encoder.ln]
        else:
            blocks = [*backbone.blocks[-config.last_blocks:], backbone.norm]
        for block in blocks:
            block.requires_grad_(True)
    return model

### Training and recovery

In [ ]:
def optimizer_for(model, config):
    groups = [{'params': model.head.parameters(), 'lr': config.head_lr}]
    parameters = [p for p in model.backbone.parameters() if p.requires_grad]
    if parameters:
        groups.append({'params': parameters, 'lr': config.backbone_lr})
    return torch.optim.AdamW(groups, weight_decay=config.weight_decay)


def trainable_state(model):
    """Frozen tensors reload from the pinned pretrained checkpoint; save changed tensors."""
    return {name: value.detach().cpu().clone() for name, value in model.named_parameters() if value.requires_grad}


def restore_trainable(model, state):
    expected = {name for name, value in model.named_parameters() if value.requires_grad}
    if set(state) != expected:
        raise ValueError('Checkpoint parameters do not match this training condition')
    model.load_state_dict(state, strict=False)


def evaluate_classifier(model, loader, device):
    model.eval()
    labels, predictions, loss_sum, count = [], [], 0., 0
    with torch.inference_mode():
        for images, targets in loader:
            logits = model(images.to(device))
            loss_sum += nn.functional.cross_entropy(logits, targets.to(device), reduction='sum').item()
            labels.extend(targets.tolist())
            predictions.extend(logits.argmax(1).cpu().tolist())
            count += len(targets)
    return {'loss': loss_sum / count, 'accuracy': float(np.mean(np.array(labels) == predictions)),
            'labels': labels, 'predictions': predictions}


def fit_support(model, config, training, development, device, checkpoint):
    """Resume at epoch boundaries; only development data choose the retained epoch."""
    optimizer = optimizer_for(model, config)
    history, best, best_state, stale, start = [], None, None, 0, 0
    checkpoint = Path(checkpoint)
    if checkpoint.exists():
        saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
        restore_trainable(model, saved['current'])
        optimizer.load_state_dict(saved['optimizer'])
        history, best, best_state = saved['history'], saved['best'], saved['best_state']
        start, stale = saved['next_epoch'], saved['stale']
        print(f'Resuming at epoch {start + 1}', flush=True)
    for epoch in range(start, config.epochs):
        if stale >= config.patience:
            break
        generator = torch.Generator().manual_seed(config.seed + epoch)
        loader = DataLoader(training, batch_size=config.batch_size, shuffle=True,
                            generator=generator, num_workers=0)
        model.train()
        # Keep pretrained BatchNorm statistics/dropout fixed for tiny support sets.
        # Evaluation mode does not disable gradients through the unfrozen block.
        model.backbone.eval()
        loss_sum, correct, count = 0., 0, 0
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = nn.functional.cross_entropy(logits, labels)
            if not torch.isfinite(loss):
                raise ValueError('Non-finite training loss')
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * len(labels)
            correct += int((logits.argmax(1) == labels).sum().item())
            count += len(labels)
        dev = evaluate_classifier(model, development, device)
        row = {'epoch': epoch + 1, 'train_loss': loss_sum / count, 'train_accuracy': correct / count,
               'dev_loss': dev['loss'], 'dev_accuracy': dev['accuracy']}
        history.append(row)
        if best is None or (row['dev_accuracy'], -row['dev_loss']) > (best['dev_accuracy'], -best['dev_loss']):
            best, best_state, stale = row, trainable_state(model), 0
        else:
            stale += 1
        saved = {'current': trainable_state(model), 'optimizer': optimizer.state_dict(),
                 'history': history, 'best': best, 'best_state': best_state,
                 'next_epoch': epoch + 1, 'stale': stale}
        atomic(checkpoint, lambda p: torch.save(saved, p))
        print(f"Epoch {epoch + 1}: train={row['train_accuracy']:.3f}, dev={row['dev_accuracy']:.3f}", flush=True)
    restore_trainable(model, best_state)
    return history, best

### Experiment runner

In [ ]:
def run_finetuning(config, storage, scratch, target='colab_gpu'):
    config.validate()
    device = check_device(target)
    storage = prepare_storage(storage)
    root, records, split, dataset_id = prepare_data(storage, scratch, config.data_config())
    trials = make_trials(records, split, config.data_config())
    manifest = {'study': 'optional-finetuning', 'version': FINETUNE_VERSION, 'config': asdict(config),
                'dataset': dataset_id, 'trials': trials, 'checkpoint': MODEL_IDS[config.model],
                'versions': environment(), 'normalization': 'L2 embeddings',
                'pretrained_batchnorm_and_dropout': 'evaluation mode in all training conditions'}
    identifier = key(manifest)
    directory = storage / 'cache/finetuning' / identifier
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / 'manifest.json', manifest)
    loaders = {part: DataLoader(Images(root, records, [(i, 0) for i in split[part]], 'none', config.seed),
                                batch_size=config.batch_size, num_workers=0)
               for part in ['dev', 'test']}
    strategies = ['frozen_head', config.strategy] if config.include_frozen_control else [config.strategy]
    rows = []
    for trial_number, trial in enumerate(trials):
        for shots in sorted(config.shots):
            original = trial[str(shots)]
            datasets = [Images(root, records, [(i, 0) for i in original], 'none', config.seed)]
            if config.views and config.policy != 'none':
                datasets.append(Images(root, records, [(i, v) for i in original for v in range(config.views)],
                                       'none' if config.policy == 'repeat' else config.policy, config.seed))
            training = ConcatDataset(datasets)
            for strategy in strategies:
                name = f'{strategy}-{shots}shot-trial{trial_number}'
                result_path = directory / f'{name}.json'
                if result_path.exists():
                    rows.append(json.loads(result_path.read_text()))
                    print('Completed fine-tuning result reused:', name, flush=True)
                    continue
                paired = replace(config, seed=config.seed + trial_number)
                started = time.perf_counter()
                model = build_adapted_model(paired, strategy, device, paired.seed)
                history, best = fit_support(model, paired, training, loaders['dev'], device, directory / f'{name}.pt')
                # The query partition is evaluated once, after development selection.
                heldout = evaluate_classifier(model, loaders['test'], device)
                result = {'model': config.model, 'strategy': strategy, 'shots': shots, 'trial': trial_number,
                          'original_support_images': len(original), 'training_rows': len(training),
                          'selected_epoch': best['epoch'], 'dev_accuracy': best['dev_accuracy'],
                          'test': score(heldout['labels'], heldout['predictions']), 'history': history,
                          'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
                          'total_parameters': sum(p.numel() for p in model.parameters()),
                          'seconds': time.perf_counter() - started, 'device': str(device)}
                write_json(result_path, result)
                rows.append(result)
                del model
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                elif device.type == 'mps':
                    torch.mps.empty_cache()
    output = storage / 'outputs/finetuning' / identifier
    output.mkdir(parents=True, exist_ok=True)
    write_json(output / 'results.json', {'manifest': manifest, 'trials': rows})
    print('Optional fine-tuning results:', output, flush=True)
    return {'output': str(output), 'manifest': manifest, 'trials': rows}


def run_finetuning_study(config, storage, scratch, target='colab_gpu'):
    """Prespecified adaptation Ã— augmentation study, with a matched repeat control."""
    config.validate()
    if config.model != 'dinov2_vits14' or not config.include_frozen_control or config.views < 1:
        raise ValueError('The DINOv2 study requires paired controls and at least one added view')
    conditions = [('none', 0), ('repeat', config.views), ('flip', config.views)]
    runs, rows = [], []
    for policy, views in conditions:
        result = run_finetuning(replace(config, policy=policy, views=views), storage, scratch, target)
        runs.append(result['manifest'])
        rows.extend({**row, 'augmentation': policy, 'added_views': views,
                     'last_blocks': config.last_blocks} for row in result['trials'])
    manifest = {'study': 'dinov2-partial-adaptation-factorial', 'config': asdict(config), 'runs': runs}
    output = Path(storage) / 'outputs/finetuning' / key(manifest)
    output.mkdir(parents=True, exist_ok=True)
    write_json(output / 'results.json', {'manifest': manifest, 'trials': rows})
    return {'output': str(output), 'manifest': manifest, 'trials': rows}

### Finetuning reporting

In [ ]:
def finetuning_contrasts(rows):
    """Paired effects: adaptation, augmentation and additional real labels."""
    indexed = {(r['strategy'], r.get('augmentation', 'single'), r['shots'], r['trial']): r for r in rows}
    contrasts = []
    def compare(kind, candidate, baseline):
        if candidate not in indexed or baseline not in indexed:
            return
        a, b = indexed[candidate], indexed[baseline]
        contrasts.append({'comparison': kind, 'strategy': candidate[0], 'augmentation': candidate[1],
                          'shots': candidate[2], 'trial': candidate[3],
                          'accuracy_gain_pp': 100 * (a['test']['accuracy'] - b['test']['accuracy']),
                          'macro_f1_gain_pp': 100 * (a['test']['macro_f1'] - b['test']['macro_f1'])})
    for strategy, policy, shots, trial in sorted(indexed):
        if strategy == 'last_block':
            compare('adaptation over frozen head', (strategy, policy, shots, trial), ('frozen_head', policy, shots, trial))
        if policy == 'flip':
            for reference in ['none', 'repeat']:
                compare('flip over ' + reference, (strategy, policy, shots, trial), (strategy, reference, shots, trial))
            if shots == 1 and indexed[strategy, policy, shots, trial].get('added_views') == 4:
                compare('1 original + 4 flips over 5 originals', (strategy, policy, shots, trial), (strategy, 'none', 5, trial))
        if policy == 'none' and shots in {5, 10}:
            compare('more originals over 1 original', (strategy, policy, shots, trial), (strategy, 'none', 1, trial))
    return contrasts


def finetuning_tables(rows):
    """Build the exported tables once for both notebook display and figures."""
    metadata = ['strategy', 'shots', 'trial', 'selected_epoch', 'trainable_parameters', 'total_parameters', 'seconds']
    metrics = ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall']
    frame = pd.DataFrame([{**{name: row[name] for name in metadata},
                           'augmentation': row.get('augmentation', 'single'),
                           **{name: row['test'][name] for name in metrics}} for row in rows])
    frame['condition'] = frame['strategy'] + ' / ' + frame['augmentation']
    groups = ['strategy', 'augmentation', 'shots']
    summary = frame.groupby(groups).agg(
        accuracy_mean=('accuracy', 'mean'), accuracy_std=('accuracy', lambda x: np.std(x, ddof=0)),
        macro_f1_mean=('macro_f1', 'mean'), macro_f1_std=('macro_f1', lambda x: np.std(x, ddof=0))).reset_index()
    contrasts = pd.DataFrame(finetuning_contrasts(rows))
    contrast_summary = contrasts.groupby(['comparison', *groups])[
        ['accuracy_gain_pp', 'macro_f1_gain_pp']].agg(['mean', 'std', 'count'])
    recall = pd.DataFrame([{'strategy': row['strategy'], 'augmentation': row.get('augmentation', 'single'),
                            'shots': row['shots'], 'trial': row['trial'], 'class': name,
                            'recall': row['test']['class_recall'][label]}
                           for row in rows for label, name in enumerate(CLASSES)])
    return {'table': frame, 'summary': summary, 'contrasts': contrasts, 'contrast_summary': contrast_summary,
            'class_recall': recall,
            'class_recall_summary': recall.groupby([*groups, 'class'])['recall'].mean().unstack('class')}


def finetuning_figures(result):
    """Export shared tables and plot actual histories, paired scores and errors."""
    output, rows = Path(result['output']), result['trials']
    tables = finetuning_tables(rows)
    exports = {'table': 'trial-metrics', 'summary': 'summary', 'contrasts': 'paired-contrasts',
               'class_recall': 'class-recall'}
    for name, filename in exports.items():
        tables[name].to_csv(output / f'{filename}.csv', index=False)
    summary = tables['summary']
    title = 'SMOKE CHECK â€” ' if result['manifest']['config']['mode'] == 'smoke' else ''
    with plt.rc_context(STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), layout='constrained')
        for (strategy, policy), group in summary.groupby(['strategy', 'augmentation']):
            for ax, metric in zip(axes, ['accuracy', 'macro_f1']):
                ax.errorbar(group['shots'], group[metric + '_mean'], yerr=group[metric + '_std'],
                            marker='o', capsize=3, label=f'{strategy} / {policy}')
                ax.set(xlabel='Labeled original images per class', ylabel=metric.replace('_', ' '),
                       ylim=(0, 1), xticks=sorted(summary.shots.unique()))
        axes[0].legend(frameon=False, fontsize=6)
        fig.suptitle(title + 'Matched neural-head control versus optional fine-tuning; mean Â± trial SD')
        save_figure(fig, output, 'fine_tuning_comparison')
        fig, axes = plt.subplots(1, 2, figsize=(8, 3.5), layout='constrained')
        maximum = max(r['shots'] for r in rows)
        for row in rows:
            if row['trial'] == 0 and row['shots'] == maximum:
                history = row['history']
                for ax, field in zip(axes, ['train_loss', 'dev_accuracy']):
                    label = row['strategy'] + ' / ' + row.get('augmentation', 'single')
                    ax.plot([h['epoch'] for h in history], [h[field] for h in history], marker='o', label=label)
                    ax.set(xlabel='Epoch', ylabel=field.replace('_', ' '))
        axes[0].legend(frameon=False, fontsize=6)
        fig.suptitle(title + f'Trial 0, {maximum}-shot; development selects epochs')
        save_figure(fig, output, 'fine_tuning_learning_curves')
        fig, axes = plt.subplots(1, 2, figsize=(10, 4.3), layout='constrained')
        for ax, strategy in zip(axes, ['frozen_head', 'last_block']):
            row = next(r for r in rows if r['strategy'] == strategy and r['trial'] == 0
                       and r['shots'] == maximum and r.get('augmentation', 'none') == 'none')
            matrix = np.array(row['test']['confusion_matrix'])
            normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
            plot = ax.imshow(normalized, vmin=0, vmax=1, cmap='Blues')
            ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)),
                   xticklabels=CLASSES, yticklabels=CLASSES,
                   xlabel='Predicted class', ylabel='True class', title=strategy)
            ax.tick_params(axis='x', labelrotation=90, labelsize=6)
            ax.tick_params(axis='y', labelsize=6)
        fig.colorbar(plot, ax=axes, label='Fraction within true class', shrink=.7)
        fig.suptitle(title + f'No augmentation, trial 0, {maximum}-shot; normalized confusion')
        save_figure(fig, output, 'fine_tuning_class_errors')
    return {**tables, 'figures': {name: str(output / f'{name}.png') for name in
            ['fine_tuning_comparison', 'fine_tuning_learning_curves', 'fine_tuning_class_errors']}}

## 4. Training protocol

`last_block` trains only the final backbone stage/block and a fresh ten-class neural head. The matched `frozen_head` control learns the same head with a frozen backbone. Both use cross-entropy and AdamW, with a smaller backbone learning rate (head: 0.001; backbone: 0.00001).

Pretrained BatchNorm statistics and backbone dropout remain in evaluation mode during adaptation, which is useful for tiny support batches. Gradients still update the selected parameters. Development accuracy selects the epoch; development loss breaks ties. The query set never chooses an epoch. Original and augmented supports stay within their original partition.

Smoke: 40 images/class, 1 trial, 2 epochs and 1 added view. `MODE = 'full'`: entire dataset, 20 trials, up to 10 epochs and 4 added views. **Mode changes the data/trial budget, never which layers are unfrozen.** The larger budget evaluates the development pool every epoch and is substantially more expensive than frozen-feature extraction.


In [ ]:
ft_config = FineTuneConfig.full(MODEL) if MODE == 'full' else FineTuneConfig(model=MODEL)
ft_config.last_blocks = LAST_BLOCKS
ft_config.validate()
device = check_device(TARGET)
print(json.dumps(asdict(ft_config), indent=2))
print('Software:', environment())


## 5. Verify the data and original-shot budgets

In [ ]:
from IPython.display import display, Image as NotebookImage, Markdown

image_root, records, split, dataset_id = prepare_data(prepare_storage(STORAGE), SCRATCH, ft_config.data_config())
supports = make_trials(records, split, ft_config.data_config())
display(pd.DataFrame([{'class': name, **{part: sum(records[i]['label'] == label for i in ids)
        for part, ids in split.items()}} for label, name in enumerate(CLASSES)]))
print('Dataset identity:', dataset_id)
print('Trial 0 original supports:', {k: len(ids) for k, ids in supports[0].items()})
example = records[split['support'][0]]
display(NotebookImage(filename=str(image_root / example['path']), width=224))
print('Unaugmented support example:', CLASSES[example['label']])


## 6. Fine-tune or resume

Each epoch saves changed tensors, optimizer state, best development weights and history atomically. A restart can repeat an incomplete epoch; completed epochs and jobs are reused. Checkpoint keys include the dataset, model, optimizer settings, budgets and software. Backbone embeddings cannot be reused during weight updates; the trained weights are the cacheable artifact.

Training checkpoints are in their own namespace and use tensor-only loading. Test evaluation happens after the best epoch is restored. This cell runs DINOv2 with its matched head-only control under none/repeat/flip support treatments.


In [ ]:
fine_tuning_results = run_finetuning_study(ft_config, STORAGE, SCRATCH, target=TARGET)


## 7. Actual results and learning curves

The plots show accuracy/Macro-F1, learning curves and a prespecified no-augmentation confusion comparison at the largest shot budget in trial 0. These matrices describe that trial only, not a selected best trial. Curves show mean ± trial standard deviation; a single smoke trial has no between-trial uncertainty estimate. Report these results separately from the frozen-logistic-regression benchmark. Compare adaptation against this notebook's matched neural-head control first. The full metrics, histories and settings are saved as JSON; summary/trial tables and PNG/PDF figures accompany them.


In [ ]:
adaptation_report = finetuning_figures(fine_tuning_results)
display(adaptation_report['summary'].round(3))
for name, filename in adaptation_report['figures'].items():
    display(Markdown('**' + name.replace('_', ' ').capitalize() + '**'))
    display(NotebookImage(filename=filename, width=900))
display(adaptation_report['table'][['strategy', 'augmentation', 'shots', 'selected_epoch',
                                    'trainable_parameters', 'total_parameters']])
display(adaptation_report['class_recall_summary'].round(3))
display(adaptation_report['contrast_summary'].round(3))
print('Artifacts:', fine_tuning_results['output'])
print('Smoke results verify execution only; positive paired gains favor the named candidate.')


## 8. Extension checklist

- [x] Keep the adaptation experiment separate from the primary frozen-backbone study.
- [x] Reuse shared partitions, nested original supports and development-only epoch selection.
- [x] Match frozen-head and adaptation head initialization, training inputs and optimizer rules.
- [x] Persist epoch checkpoints and restore the selected weights before query evaluation.
- [x] Select DINOv2 last-block adaptation and define paired none/repeat/flip comparisons.
- [ ] Lock learning rates, epoch budget and any two-block ablation before final query evaluation.
- [ ] Run the agreed full grid on the intended GPU and review class-specific errors.
- [ ] Report extra development labels, training cost and differences from the primary logistic-regression protocol.
